# Nov 1 Task Spaceship Titanic Feature Engineering and Model Comparison

## Goal

Over the weekend, I did some experimentation with the baseline and was able to get a higher Kaggle score using feature engineering and model comparison. In this notebook, you will recreate that process and see which approach performs best on your validation data.

In this notebook, you will recreate the process that produced the stronger submission. You will practice:

- creating features from existing columns
- preparing numerical and categorical features
- comparing models using validation accuracy
- creating a Kaggle submission

The notebook includes instructions, hints, and marked sections where you need to complete the code.

## Before you begin

1. Make your own copy of this notebook.
2. Upload `train.csv`, `test.csv`, and `sample_submission.csv` to one folder in Google Drive.
3. Update the `DATA_DIR` path below.
4. Run the cells from top to bottom.

Do not edit the shared master notebook directly. Work in your own Colab copy or GitHub branch.

## Helpful resources before you start

You do not need to memorize these methods. Use these explanations and examples while completing the notebook.

### Feature engineering
Feature engineering means creating a new column from information that already exists. For example, we can combine five spending columns into one total:

```python
dataframe["TotalSpending"] = dataframe[spending_columns].sum(axis=1, min_count=1)
```

`axis=1` means that pandas calculates across each row.

### Splitting text columns
The `Cabin` values look similar to `A/12/P`. We can split the text at each slash:

```python
dataframe["CabinDeck"] = dataframe["Cabin"].str.split("/").str[0]
dataframe["CabinSide"] = dataframe["Cabin"].str.split("/").str[2]
```

### Counting groups
`value_counts()` counts how often each value appears. `map()` uses those counts to create a new column:

```python
group_id_counts = group_ids.value_counts()
dataframe["GroupSize"] = dataframe["GroupId"].map(group_id_counts)
```

### Handling missing values and categories
The numerical pipeline fills missing numerical values with the median. The categorical pipeline fills missing categories with the most common value, then uses `OneHotEncoder` to convert categories into numbers.

### Comparing models
Validation accuracy measures how many validation predictions were correct. We will compare logistic regression and Random Forest using the same validation rows. Do not choose a model based only on the name of the algorithm; compare the validation results.

## 1. Connect Google Drive and load the data

In [3]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Change this path to the folder containing your CSV files.
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"

Mounted at /content/drive


In [4]:
from pathlib import Path
import pandas as pd

data_folder = Path(DATA_DIR)
train = pd.read_csv(data_folder / "train.csv")
test = pd.read_csv(data_folder / "test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
display(train.head())

Training data shape: (8693, 14)
Test data shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


### Note about loading the data

In the earlier shared baseline notebook, we loaded files that had been uploaded directly into the temporary Colab session using paths such as:

```python
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")
sample_submission = pd.read_csv("/content/sample_submission.csv")
```

This notebook uses a Google Drive folder instead:

```python
train = pd.read_csv(data_folder / "train.csv")
test = pd.read_csv(data_folder / "test.csv")
```

The Drive method takes one extra setup step, but the files remain available in your Drive. The original `/content/...` method is also correct if you upload the CSV files directly into the current Colab session. The important part is that the file paths match where your files are stored.

## 2. Import the machine learning tools

The preprocessing tools will help us fill missing values and convert categorical columns into a format that models can use.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 3. Create a consistent validation split

We will use the same validation passengers for every model. This makes the accuracy comparison fair.

In [6]:
target = "Transported"
y = train[target].astype(int)

train_indices, valid_indices = train_test_split(
    train.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(train_indices))
print("Validation rows:", len(valid_indices))

Training rows: 6954
Validation rows: 1739


## 4. Create engineered features

Feature engineering means creating useful new columns from information that already exists. We will create five features:

- `TotalSpending`: the total amount spent across the five services
- `CabinDeck`: the first part of the cabin value
- `CabinSide`: the last part of the cabin value
- `GroupSize`: the number of passengers in the same group
- `IsAlone`: whether the passenger is the only person in their group

### Your task

Complete the marked code below. The hints explain what each line should do.

In [7]:
train_engineered = train.copy()
test_engineered = test.copy()

spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

# TODO 1: add the five spending columns together across each row.
# Hint: use .sum(axis=1, min_count=1).
train_engineered["TotalSpending"] = train_engineered[spending_columns].sum(axis=1, min_count=1)
test_engineered["TotalSpending"] = test_engineered[spending_columns].sum(axis=1, min_count=1)


# TODO 2: extract the first section of Cabin into CabinDeck.
# Hint: Cabin values look like A/12/P. Use .str.split("/").str[0].
train_engineered["CabinDeck"] = train_engineered["Cabin"].str.split("/").str[0]
test_engineered["CabinDeck"] = test_engineered["Cabin"].str.split("/").str[0]

# TODO 3: extract the last section of Cabin into CabinSide.
# Hint: use .str.split("/").str[2].
train_engineered["CabinSide"] = train_engineered["Cabin"].str.split("/").str[2]
test_engineered["CabinSide"] = test_engineered["Cabin"].str.split("/").str[2]

### Group features

The first part of `PassengerId` identifies a passenger group. For example, passengers `1234_01` and `1234_02` belong to the same group.

Complete the next cell using the hints.

In [12]:
# TODO 4: create GroupId from the part of PassengerId before the underscore.
# Hint: use .str.split("_").str[0].
train_engineered["GroupId"] = train_engineered["PassengerId"].str.split("_").str[0]
test_engineered["GroupId"] = test_engineered["PassengerId"].str.split("_").str[0]

# This combines both datasets so group sizes include all passengers.
all_passenger_ids = pd.concat([
    train_engineered["PassengerId"],
    test_engineered["PassengerId"]
])

all_group_ids = all_passenger_ids.str.split("_").str[0]
group_id_counts = all_group_ids.value_counts()

# TODO 5: map each GroupId to its group size.
train_engineered["GroupSize"] = train_engineered["GroupId"].map(group_id_counts)
test_engineered["GroupSize"] = test_engineered["GroupId"].map(group_id_counts)

# TODO 6: create IsAlone. It should equal 1 when GroupSize is 1.
train_engineered["IsAlone"] = train_engineered["GroupSize"] == 1
test_engineered["IsAlone"] = test_engineered["GroupSize"] == 1

## 5. Check your engineered features

Run this cell after completing the previous two cells. If you get a `KeyError`, check that the column names match the original dataset exactly.

In [13]:
new_columns = [
    "TotalSpending",
    "CabinDeck",
    "CabinSide",
    "GroupSize",
    "IsAlone"
]

display(train_engineered[new_columns].head())
print(train_engineered[new_columns].dtypes)
print("Feature creation check passed.")

,TotalSpending,CabinDeck,CabinSide,GroupSize,IsAlone
0,0.0,B,P,1,True
1,736.0,F,S,1,True
2,10383.0,A,S,2,False
3,5176.0,A,S,2,False
4,1091.0,F,S,1,True


TotalSpending    float64
CabinDeck         object
CabinSide         object
GroupSize          int64
IsAlone             bool
dtype: object
Feature creation check passed.


## 6. Build the preprocessing and model function

The function below will be used to build both logistic regression and Random Forest models.

### Your task

Complete the categorical pipeline. It needs to:

1. Fill missing categorical values using the most frequent value.
2. Convert categorical values into numbers using `OneHotEncoder`.

The numerical pipeline is already provided as an example.

In [17]:
def build_model(categorical_features, numerical_features, classifier):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    # TODO 7: create the categorical pipeline.
    # It should contain a most_frequent SimpleImputer and
    # a OneHotEncoder(handle_unknown="ignore").
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
         ("onehot", OneHotEncoder(handle_unknown="ignore"))])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numerical_features),
        # TODO 8: add the categorical transformer here.
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

## 7. Choose the input features

We will not use `PassengerId`, `Name`, `Cabin`, or `GroupId` directly. They are identifiers or contain information that we have already transformed into more useful features.

In [18]:
drop_columns = [target, "PassengerId", "Name", "Cabin", "GroupId"]

categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide"
]

numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "TotalSpending",
    "GroupSize",
    "IsAlone"
]

X_engineered = train_engineered.drop(columns=drop_columns)

print("Model input columns:", list(X_engineered.columns))

Model input columns: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpending', 'CabinDeck', 'CabinSide', 'GroupSize', 'IsAlone']


## 8. Train and compare two models

We will compare logistic regression and Random Forest using the engineered features. Both models use the same preprocessing and the same validation split.

### Your task

Complete the two model definitions. Use:

- `LogisticRegression(max_iter=1000)`
- `RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1)`

In [21]:
# TODO 9: create the logistic regression model by calling build_model.
logistic_model = build_model(
    categorical_features, numerical_features,
    LogisticRegression(max_iter=1000)
)

# TODO 10: create the Random Forest model by calling build_model.
random_forest_model = build_model(
    categorical_features, numerical_features,
    RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1)
)

## 9. Train and evaluate the models

Use the training indices to fit each model and the validation indices to calculate accuracy.

In [26]:
model_results = {}

# TODO 11: fit logistic_model using the training rows.
logistic_model.fit(X_engineered.loc[train_indices], y.loc[train_indices])

# TODO 12: create logistic regression predictions for validation rows.
logistic_predictions = logistic_model.predict(X_engineered.loc[valid_indices])

# TODO 13: calculate logistic regression accuracy.
logistic_accuracy = accuracy_score(y.loc[valid_indices], logistic_predictions)

# TODO 14: repeat the three steps above for random_forest_model.
random_forest_model.fit(X_engineered.loc[train_indices], y.loc[train_indices])
random_forest_predictions = random_forest_model.predict(X_engineered.loc[valid_indices])
random_forest_accuracy = accuracy_score(y.loc[valid_indices], random_forest_predictions)

# After completing the TODOs, uncomment and run these lines:
model_results["engineered_logistic"] = logistic_accuracy
model_results["engineered_random_forest"] = random_forest_accuracy
results = pd.DataFrame({"Model": model_results.keys(), "Validation Accuracy": model_results.values()})
results = results.sort_values("Validation Accuracy", ascending=False).reset_index(drop=True)
display(results)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Validation Accuracy
0,engineered_random_forest,0.805060
1,engineered_logistic,0.790109


## 10. Create a submission from the better model

After comparing the validation accuracies, choose the model with the higher accuracy. Retrain that model using all labeled training data, then use it to predict the test data.

In [29]:
# TODO 15: set best_model to the model with the higher validation accuracy.
best_model = random_forest_model
best_model_name = "engineered_random_forest"

# TODO 16: fit best_model using all X_engineered and y.

test_drop_columns = [column for column in drop_columns if column in test_engineered.columns]
test_input = test_engineered.drop(columns=test_drop_columns)

# TODO 17: create predictions for test_input.
test_predictions = best_model.predict(test_input).astype(bool)

# TODO 18: create a DataFrame with PassengerId and Transported.
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_predictions
})

## 11. Check and save the submission

Run this after completing the previous section.

In [30]:
print("Submission shape:", submission.shape)
print("Expected test rows:", len(test))
print("Submission columns:", list(submission.columns))
print("Missing predictions:", submission["Transported"].isna().sum())

assert len(submission) == len(test)
assert list(submission.columns) == ["PassengerId", "Transported"]
assert submission["Transported"].isna().sum() == 0

submission_path = data_folder / "nov_5_model_submission.csv"
submission.to_csv(submission_path, index=False)
print("Submission checks passed.")
print("Saved to:", submission_path)

Submission shape: (4277, 2)
Expected test rows: 4277
Submission columns: ['PassengerId', 'Transported']
Missing predictions: 0
Submission checks passed.
Saved to: /content/drive/MyDrive/Colab Notebooks/nov_1_model_submission.csv


## 12. Final checklist

Before the next meeting, make sure you have:

- created all five engineered features
- completed the categorical preprocessing pipeline
- trained both logistic regression and Random Forest
- recorded both validation accuracies:** 0.805060, 0.790109**
- selected the better model: **engineered_random_forest**
- created and checked `nov_1_model_submission.csv` **(renamed to nov_5_model_submission.csv)**
- uploaded the submission to Kaggle if possible: **kaggle score 0.79471**

The goal is not only to obtain a higher score. It is also to understand how feature engineering, preprocessing, model selection, and validation work together.